pip install apryse-sdk --extra-index-url=https://pypi.apryse.com

In [2]:
# Import Libraries
import os
import sys
from apryse_sdk import PDFDoc, Optimizer, SDFDoc, PDFNet

#from PDFNetPython3.PDFNetPython import PDFDoc, Optimizer, SDFDoc, PDFNet

In [3]:
def get_size_format(b, factor=1024, suffix="B"):
    """
    Converte um tamanho em bytes para um formato legível (KB, MB, GB, etc.).
    
    Parâmetros:
        b (int): Tamanho em bytes.
        factor (int, opcional): Fator de conversão (padrão: 1024 para binário).
        suffix (str, opcional): Sufixo da unidade (padrão: "B" para Bytes).
    
    Retorna:
        str: String formatada com o tamanho ajustado (ex: "1.20MB").
    
    Exemplo:
        1253656 => '1.20MB'
        1253656678 => '1.17GB'
    """
    for unit in ["", "K", "M", "G", "T", "P", "E", "Z"]:
        if b < factor:
            return f"{b:.2f}{unit}{suffix}"
        b /= factor
    return f"{b:.2f}Y{suffix}"

In [4]:
def compress_file(input_file: str, output_file: str):
    """
    Comprime um arquivo PDF usando a biblioteca PDFNet, reduzindo seu tamanho.
    
    Parâmetros:
        input_file (str): Caminho do arquivo PDF de entrada.
        output_file (str): Caminho do arquivo PDF de saída (se vazio, sobrescreve o original).
    
    Retorna:
        bool: True se a compressão foi bem-sucedida, False caso contrário.
    """
    if not output_file:
        output_file = input_file
    
    initial_size = os.path.getsize(input_file)  # Tamanho original do arquivo
    
    try:
        # Inicializa a biblioteca PDFNet com uma chave de licença (demo)
        PDFNet.Initialize("demo:1652721067096:7b89dd1603000000008ca8f036d3c1112cd0debd5cca3f78321ecf6f88")
        doc = PDFDoc(input_file)  # Carrega o PDF
        
        # Configura o manipulador de segurança padrão (necessário para edição)
        doc.InitSecurityHandler()
        
        # Otimiza o PDF: remove metadados redundantes e compacta fluxos de dados
        Optimizer.Optimize(doc)
        
        # Salva o arquivo compactado (e_linearized para web)
        doc.Save(output_file, SDFDoc.e_linearized)
        doc.Close()  # Fecha o documento
        
    except Exception as e:
        print("Error compress_file=", e)
        doc.Close()
        return False
    
    # Calcula estatísticas pós-compressão
    compressed_size = os.path.getsize(output_file)
    ratio = 1 - (compressed_size / initial_size)  # Taxa de compressão (0-1)
    
    # Resumo da operação
    summary = {
        "Input File": input_file,
        "Initial Size": get_size_format(initial_size),
        "Output File": output_file,
        "Compressed Size": get_size_format(compressed_size),
        "Compression Ratio": "{0:.3%}.".format(ratio)  # Formata como porcentagem
    }
    
    # Exibe o resumo formatado
    print("--------------------- Resumo da compressão ------------------------")
    print("\n".join("{}:{}".format(i, j) for i, j in summary.items()))
    print("-------------------------------------------------------------------")
    print("")
    return True


In [5]:
# Pasta contendo os PDFs a serem comprimidos (substitua pelo seu caminho)
input_folder = r"C:\Users\Rafael Bruno\Downloads\\"

# Percorre todos os arquivos PDF na pasta e subpastas
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith(".pdf"):
            pdf_path = os.path.join(root, file)  # Caminho completo do PDF
            compress_file(pdf_path, pdf_path)  # Comprime o arquivo (sobrescreve o original)

print("Otimização concluída!")

--------------------- Resumo da compressão ------------------------
Input File:C:\Users\Rafael Bruno\Downloads\\Geometria Espacial.pdf
Initial Size:70.62MB
Output File:C:\Users\Rafael Bruno\Downloads\\Geometria Espacial.pdf
Compressed Size:11.21MB
Compression Ratio:84.125%.
-------------------------------------------------------------------

--------------------- Resumo da compressão ------------------------
Input File:C:\Users\Rafael Bruno\Downloads\\Hidrostática.pdf
Initial Size:25.18MB
Output File:C:\Users\Rafael Bruno\Downloads\\Hidrostática.pdf
Compressed Size:6.36MB
Compression Ratio:74.756%.
-------------------------------------------------------------------

--------------------- Resumo da compressão ------------------------
Input File:C:\Users\Rafael Bruno\Downloads\\Termodinâmica.pdf
Initial Size:38.19MB
Output File:C:\Users\Rafael Bruno\Downloads\\Termodinâmica.pdf
Compressed Size:6.71MB
Compression Ratio:82.440%.
-------------------------------------------------------------